# Cool T-Shirts Website Funnel Analysis — Solution Notebook

**Extended project based on Codecademy-style “Joining Tables in R” + Funnel for Cool T-Shirts**

This notebook walks through a complete multi-table funnel analysis using `dplyr` joins.  
It includes the core lesson exercises, alternate implementations, extra practice, a parameterised Monte-Carlo simulation, and audience-adapted interpretation notes.

---

## Learning objectives
- Load and inspect related tables that share a key (`user_id`)
- Perform `inner_join`, `left_join` (and know when to use `full_join` / `right_join`)
- Compute stage conversion rates and time-between-events metrics
- Visualise a classic marketing/product funnel
- Explore robustness via simulation and alternate base-R syntax

---

## Process flowchart

![Cool T-Shirts Funnel Flowchart](cool_tshirts_funnel_flowchart.png)

*The flowchart above is also included in the Practice Skeleton so you can keep the big picture in view while coding.*

## 0. Setup

Load the tidyverse packages we need.  `lubridate` makes datetime arithmetic clean; `ggplot2` is used for the funnel and time plots.

In [ ]:
# Solution — packages
library(readr)
library(dplyr)
library(lubridate)
library(ggplot2)
library(tidyr)   # for complete() if needed in practice

# For reproducible simulation later
set.seed(42)

## 1. Load & inspect the three tables

Cool T-Shirts tracks three events on their site:

| Table        | Meaning                                      | Key columns              |
|--------------|----------------------------------------------|--------------------------|
| `visits`     | Every landing-page visit                     | `user_id`, `visit_time`  |
| `checkouts`  | Users who reached the checkout page          | `user_id`, `checkout_time` |
| `purchases`  | Users who completed a purchase               | `user_id`, `purchase_time` |

**Checkpoint style questions (from the original lesson):**
1. How many unique users visited?
2. Which columns are shared across tables?
3. Are the timestamp columns already POSIXct / Date?

In [ ]:
# Solution — load data (adjust path if running from a different working directory)
visits    <- read_csv("data/visits.csv",    show_col_types = FALSE)
checkouts <- read_csv("data/checkouts.csv", show_col_types = FALSE)
purchases <- read_csv("data/purchases.csv", show_col_types = FALSE)

# Quick inspection
cat("=== visits ===\n")
print(head(visits, 3))
cat("\nRows:", nrow(visits), " | Unique users:", n_distinct(visits$user_id), "\n")

cat("\n=== checkouts ===\n")
print(head(checkouts, 3))
cat("\nRows:", nrow(checkouts), " | Unique users:", n_distinct(checkouts$user_id), "\n")

cat("\n=== purchases ===\n")
print(head(purchases, 3))
cat("\nRows:", nrow(purchases), " | Unique users:", n_distinct(purchases$user_id), "\n")

# Column types
glimpse(visits)
glimpse(checkouts)
glimpse(purchases)

**Expected output (with the supplied synthetic data):**  
- 250 visits, 100 checkouts, 25 purchases  
- All tables share `user_id` (integer)  
- Times are parsed as `<dttm>` by `readr`

---

## 2. Visit → Checkout join & time-to-checkout

We only care about users who **both** visited **and** checked out → classic **inner join**.

Then compute the elapsed hours between the two events.

In [ ]:
# Solution — inner join visits + checkouts
v_to_c <- visits %>%
  inner_join(checkouts, by = "user_id")

# Elapsed time in hours
v_to_c <- v_to_c %>%
  mutate(
    time_to_checkout_hrs = as.numeric(difftime(checkout_time, visit_time, units = "hours"))
  )

cat("Rows after inner join:", nrow(v_to_c), "\n")
summary(v_to_c$time_to_checkout_hrs)

cat("\nMean time to checkout (hours):", mean(v_to_c$time_to_checkout_hrs), "\n")
cat("Median time to checkout (hours):", median(v_to_c$time_to_checkout_hrs), "\n")

**Expected:** 100 rows, mean ≈ 37.0 h, median ≈ 39.2 h.

### Alternate syntax (base R `merge`)

Same result without the pipe:

In [ ]:
# Alternate — base::merge
v_to_c_base <- merge(visits, checkouts, by = "user_id")
v_to_c_base$time_to_checkout_hrs <- as.numeric(
  difftime(v_to_c_base$checkout_time, v_to_c_base$visit_time, units = "hours")
)
identical(sort(v_to_c$user_id), sort(v_to_c_base$user_id))  # should be TRUE

---

## 3. Full funnel table (left joins keep every visit)

To calculate conversion rates we need the **denominator** (all visits).  Therefore we use successive **left joins** so that users who never checked out or purchased still appear (with `NA` in the later columns).

In [ ]:
# Solution — build the complete funnel table
funnel <- visits %>%
  left_join(checkouts, by = "user_id") %>%
  left_join(purchases, by = "user_id") %>%
  mutate(
    reached_checkout = !is.na(checkout_time),
    made_purchase    = !is.na(purchase_time),
    time_to_checkout_hrs = as.numeric(difftime(checkout_time, visit_time, units = "hours")),
    time_visit_to_purchase_hrs = as.numeric(difftime(purchase_time, visit_time, units = "hours"))
  )

cat("Funnel table dimensions:", dim(funnel), "\n")
head(funnel, 4)

---

## 4. Core funnel metrics

Classic three-stage funnel numbers and conversion rates.

In [ ]:
# Solution — funnel counts & rates
n_visits    <- nrow(funnel)
n_checkouts <- sum(funnel$reached_checkout)
n_purchases <- sum(funnel$made_purchase)

conv_visit_to_checkout   <- n_checkouts / n_visits
conv_checkout_to_purchase <- n_purchases / n_checkouts
conv_overall              <- n_purchases / n_visits

funnel_summary <- tibble(
  stage = c("Visits", "Checkouts", "Purchases"),
  count = c(n_visits, n_checkouts, n_purchases),
  conversion_from_previous = c(NA, conv_visit_to_checkout, conv_checkout_to_purchase),
  conversion_from_visit    = c(1, conv_visit_to_checkout, conv_overall)
)

print(funnel_summary)

cat("\n--- Key rates ---\n")
cat(sprintf("Visit → Checkout:   %.1f%%\n", 100 * conv_visit_to_checkout))
cat(sprintf("Checkout → Purchase: %.1f%%\n", 100 * conv_checkout_to_purchase))
cat(sprintf("Overall (Visit → Purchase): %.1f%%\n", 100 * conv_overall))

**With the supplied data:**  
- Visit → Checkout ≈ 40 %  
- Checkout → Purchase = 25 %  
- Overall ≈ 10 %

---

## 5. Visualisations

### 5a. Classic funnel bar chart

In [ ]:
# Solution — funnel bar chart
funnel_plot_df <- funnel_summary %>%
  mutate(stage = factor(stage, levels = c("Visits", "Checkouts", "Purchases")))

ggplot(funnel_plot_df, aes(x = stage, y = count, fill = stage)) +
  geom_col(width = 0.6) +
  geom_text(aes(label = count), vjust = -0.4, size = 4) +
  scale_fill_manual(values = c("#3498db", "#e67e22", "#27ae60")) +
  labs(title = "Cool T-Shirts Website Funnel",
       subtitle = "Absolute counts at each stage",
       x = NULL, y = "Number of users") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

### 5b. Time-to-checkout distribution

In [ ]:
# Solution — histogram of time to checkout
ggplot(v_to_c, aes(x = time_to_checkout_hrs)) +
  geom_histogram(bins = 20, fill = "#16a085", colour = "white") +
  geom_vline(xintercept = mean(v_to_c$time_to_checkout_hrs),
             colour = "red", linetype = "dashed", linewidth = 1) +
  labs(title = "Time from Visit to Checkout",
       subtitle = "Red dashed line = mean",
       x = "Hours", y = "Count of users") +
  theme_minimal(base_size = 13)

---

## 6. Alternate join strategies & diagnostics

### 6a. What if we had used an inner join all the way?

We would lose the non-converting visitors and could no longer compute true conversion rates from the visit base.

In [ ]:
# Alternate — pure inner-join funnel (loses non-converters)
funnel_inner_only <- visits %>%
  inner_join(checkouts, by = "user_id") %>%
  inner_join(purchases, by = "user_id")

cat("Rows if we inner-join everything:", nrow(funnel_inner_only),
    "(only the final purchasers) — cannot compute visit-based rates!\n")

### 6b. Full join (useful when tables may contain “orphans”)

In real data you sometimes see a checkout without a recorded visit (tracking bugs).  A `full_join` surfaces those anomalies.

In [ ]:
# Alternate — full_join to detect mismatches
full_vc <- visits %>%
  full_join(checkouts, by = "user_id")

orphan_checkouts <- full_vc %>% filter(is.na(visit_time))
orphan_visits    <- full_vc %>% filter(is.na(checkout_time))

cat("Checkouts without a visit record:", nrow(orphan_checkouts), "\n")
cat("Visits that never checked out:", nrow(orphan_visits), "\n")

---

## 7. More practice exercises

### Practice A — Add a calendar month and look at monthly conversion

In [ ]:
# More practice A — monthly conversion
funnel_month <- funnel %>%
  mutate(month = floor_date(visit_time, "month")) %>%
  group_by(month) %>%
  summarise(
    visits    = n(),
    checkouts = sum(reached_checkout),
    purchases = sum(made_purchase),
    conv_vc   = checkouts / visits,
    conv_cp   = purchases / checkouts,
    .groups = "drop"
  )

print(funnel_month)

### Practice B — `bind_rows` for multi-period data

Imagine you receive separate CSVs for June and July.  Concatenate them first, then analyse.

In [ ]:
# More practice B — demonstrate bind_rows (here we just split the existing data)
june_visits  <- visits %>% filter(month(visit_time) == 6)
july_visits  <- visits %>% filter(month(visit_time) == 7)  # may be empty in this synthetic set

all_visits_bound <- bind_rows(june_visits, july_visits)
cat("Rows after bind_rows:", nrow(all_visits_bound), "(should equal nrow(visits) if both months present)\n")

### Practice C — Identify the “fastest” converters

Which users checked out in under 2 hours?

In [ ]:
# More practice C — fast converters
fast <- v_to_c %>%
  filter(time_to_checkout_hrs < 2) %>%
  arrange(time_to_checkout_hrs)

cat("Users who checked out in < 2 hours:", nrow(fast), "\n")
head(fast, 5)

---

## 8. Simulation section (parameterised)

Change the conversion probabilities and time-noise parameters below and re-run the cell to see how the funnel metrics and average times respond.  This is a lightweight Monte-Carlo that helps product managers understand sensitivity.

In [ ]:
# ============================================================
# SIMULATION — modify these parameters and re-run
# ============================================================
n_sim_users      <- 500          # size of each synthetic cohort
p_checkout       <- 0.40         # probability a visitor reaches checkout
p_purchase_given_checkout <- 0.25  # probability a check-out becomes a purchase
mean_hours_to_checkout <- 36     # expected hours from visit to checkout
sd_hours_to_checkout   <- 20
n_monte_carlo    <- 200          # number of simulated cohorts

# ------------------------------------------------------------
# Helper that returns one simulated funnel summary
# ------------------------------------------------------------
simulate_one_funnel <- function(n, p_c, p_p, mu_h, sd_h) {
  uid <- seq_len(n)
  # visit times (arbitrary)
  visit_t <- as.POSIXct("2023-06-01") + runif(n, 0, 30*24*3600)
  # who checks out?
  checkout_flag <- runif(n) < p_c
  checkout_t <- visit_t + abs(rnorm(n, mu_h, sd_h)) * 3600
  checkout_t[!checkout_flag] <- NA
  # who purchases among those who checked out?
  purchase_flag <- checkout_flag & (runif(n) < p_p)
  purchase_t <- checkout_t + abs(rnorm(n, 6, 4)) * 3600
  purchase_t[!purchase_flag] <- NA

  n_c <- sum(checkout_flag)
  n_p <- sum(purchase_flag)
  times <- as.numeric(difftime(checkout_t[checkout_flag], visit_t[checkout_flag], units = "hours"))

  tibble(
    n_visits    = n,
    n_checkouts = n_c,
    n_purchases = n_p,
    conv_vc     = n_c / n,
    conv_cp     = ifelse(n_c > 0, n_p / n_c, NA),
    conv_overall = n_p / n,
    mean_time_h = mean(times, na.rm = TRUE),
    median_time_h = median(times, na.rm = TRUE)
  )
}

# Run the Monte-Carlo
sim_results <- bind_rows(lapply(1:n_monte_carlo, function(i) {
  simulate_one_funnel(n_sim_users, p_checkout, p_purchase_given_checkout,
                      mean_hours_to_checkout, sd_hours_to_checkout)
}))

cat("=== Simulation summary (means across", n_monte_carlo, "cohorts) ===\n")
print(summarise(sim_results,
                mean_conv_vc = mean(conv_vc),
                mean_conv_cp = mean(conv_cp, na.rm = TRUE),
                mean_overall = mean(conv_overall),
                mean_of_mean_time = mean(mean_time_h, na.rm = TRUE)))

# Distribution of overall conversion
ggplot(sim_results, aes(x = conv_overall)) +
  geom_histogram(bins = 25, fill = "#9b59b6", colour = "white") +
  geom_vline(xintercept = mean(sim_results$conv_overall), colour = "red", linetype = "dashed") +
  labs(title = "Monte-Carlo: Overall Conversion Rate Distribution",
       subtitle = paste0("p_checkout = ", p_checkout, ", p_purchase|checkout = ", p_purchase_given_checkout),
       x = "Overall conversion (visit → purchase)", y = "Count of simulated cohorts") +
  theme_minimal(base_size = 13)

**Try changing** `p_checkout` to 0.55 or `p_purchase_given_checkout` to 0.15 and re-run.  Observe how the histogram shifts and how the mean time is relatively stable (it only depends on the time-generating parameters).

---

## 9. Audience-adapted takeaways

Drawing on the attached audience-analysis material:

| Audience type | What they care about | How we present the funnel |
|---------------|----------------------|---------------------------|
| **Data analyst / technician** | Exact rates, join logic, time distributions, edge cases (orphans) | Full notebook, code, diagnostics, simulation |
| **Product / growth executive** | “Where do we lose users?”, headline conversion, impact of a +5 pp lift | 1-page summary + simple funnel chart + 1–2 sensitivity numbers |
| **Nonspecialist / board** | Big picture only, plain language, no jargon | “Out of 100 visitors, 40 start checkout, 10 finish a purchase. Average time to checkout is ~1.5 days.” |

Key message for Cool T-Shirts leadership:  
> The biggest drop-off is between visit and checkout (60 % leave).  Improving the product-page experience or simplifying the path to cart is likely higher-leverage than optimising the final payment step (already 25 % conversion).

---

## End of Solution Notebook

You now have a complete, reusable pattern for any multi-stage event funnel that lives in separate tables.